# iterative-probe — 2×2 の推移

変調範囲（`fc` / `stage4+fc`）× GroupDRO step size（1e-3 / 1e-2）の 4 run を読む。
seed 42、warmup 2 epoch → stage01 5 epoch → stage02 5 epoch、cohort 10 クラスタ。
run-id と条件の対応は [runs.md](runs.md)。

**図は変調範囲ごとに分ける。** アーキテクチャを図の単位に置くことで、線の色は step size だけを
表す。4 条件を 1 枚に重ねると線が交差して読めない。

run 記録の読み込みと集計は `collect.py` が持ち、この notebook は集計済みの表を読んで
分析と可視化だけを行う。

```bash
uv run python analysis/iterative-probe/collect.py <run-id> <run-id> <run-id> <run-id>
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.style
import numpy as np
import pandas as pd

ROOT = Path.cwd() if Path("results").is_dir() else Path("analysis/iterative-probe")
RESULTS, FIGURES = ROOT / "results", ROOT / "figures"
FIGURES.mkdir(exist_ok=True)

# 旧 repo から持ってきた図 style。Okabe–Ito の色循環と論文向けの軸設定を持つ。
matplotlib.style.use(ROOT.parent / "styles" / "fairness.mplstyle")
MUTED, GRID = "#52514e", "#dcdcd8"

frame = pd.read_csv(RESULTS / "epoch_metrics.csv")

# 色は step size に固定する。変調範囲は図そのものが表すので、色には載せない。
STEP_COLOR = {0.001: "#0173B2", 0.01: "#DE8F05"}
MODULATIONS = ["fc", "stage4+fc"]
STEPS = sorted(frame["step_size"].unique())

frame.groupby(["modulation", "step_size"])["run_id"].agg(["first", "count"])

## 指標の読み方

hidden cohort 系の指標は名前が似ているので、`hidden_cohort_logger.py:301` の定義で揃えておく。

| 列 | 定義 | 図の見出し |
|---|---|---|
| `val/hidden_min_auroc` | 10 cohort の AUROC の**最小**。いちばん悪い群の性能 | Worst-group AUROC |
| `val/hidden_auroc_gap` | 同じく**最大 − 最小**。群間がどれだけ開いているか | Cohort AUROC spread |
| `val/hidden_loss_gap` | cohort 平均 loss の最大 − 最小 | （図にしていない） |

spread は worst group の値ではなく**散らばりの幅**なので、小さいほど群間が揃っている。
worst が下がっても best がもっと下がれば spread は縮むため、2 つは別々に読む。

## 到達点

各 run の最終 epoch。`hidden_*` は cohort が存在する stage にだけあるので、warmup では空になる。

**注意**: cohort は stage ごとに、さらに run ごとに引き直される。hidden group は run 間でも
stage 間でも別物なので、群の同一性を前提にした読み方はできない。

In [ ]:
COLUMNS = {
    "val/auroc": "global AUROC",
    "val/bacc": "global bACC",
    "val/hidden_min_auroc": "worst AUROC",
    "val/hidden_min_bacc": "worst bACC",
    "val/hidden_auroc_gap": "cohort AUROC spread",
    "train/group_dro/weight_entropy": "weight entropy",
    "train/group_dro/max_q": "max q",
}

last = frame.sort_values("run_epoch").groupby(["modulation", "step_size"]).tail(1)
last = last.set_index([last["modulation"], last["step_size"]])[list(COLUMNS)].rename(columns=COLUMNS)
last.index.names = ["modulation", "step size"]
last.round(4)

## 描画の共通部分

横軸は run 全体の通し epoch。縦の区切りは stage の境目で、そこで cohort が引き直され、
GroupDRO の `q` も初期化される。seed は 42 の 1 本だけなので、旧 repo の図のような
seed 間のばらつき帯は引けない。

In [ ]:
def panel(axis, modulation, column, title, reference=None, reference_label=""):
    """1 つの panel に、指定した変調範囲の step size 別の推移を描く。

    Args:
        axis: 描画先
        modulation: 描く変調範囲
        column: 描く列
        title: panel の見出し
        reference: 水平の基準線。不要なら None
        reference_label: 基準線に添える名前

    Returns:
        None
    """
    data = frame[frame["modulation"] == modulation]
    if reference is not None:
        axis.axhline(reference, color=MUTED, linewidth=0.8, linestyle="--", zorder=1)
        axis.text(data["run_epoch"].max() + 0.3, reference, reference_label, color=MUTED, fontsize=8, va="center")
    for _, group in data.groupby("stage_index"):
        left = group["run_epoch"].min()
        if left > 0:
            axis.axvline(left - 0.5, color=GRID, linewidth=0.8, zorder=0)
    for step in STEPS:
        series = data[data["step_size"] == step]
        values = series[column]
        axis.plot(series["run_epoch"], values, color=STEP_COLOR[step], label=f"step {step:g}", zorder=3)
        valid = values.dropna()
        if not valid.empty:
            axis.annotate(f"{valid.iloc[-1]:.3f}", (series.loc[valid.index[-1], "run_epoch"], valid.iloc[-1]), textcoords="offset points", xytext=(6, 0), va="center", color=MUTED, fontsize=8)
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.set_xlim(-0.5, data["run_epoch"].max() + 2.0)


def figure_for(modulation, panels, columns=3):
    """変調範囲 1 つ分の figure を、指標ごとの panel を並べて作る。

    Args:
        modulation: 描く変調範囲
        panels: `(列名, 見出し)` または `(列名, 見出し, 基準線, 基準線の名前)` の並び
        columns: 1 行あたりの panel 数

    Returns:
        Figure: 描画した figure
    """
    rows = -(-len(panels) // columns)
    figure, axes = plt.subplots(rows, columns, figsize=(4.4 * columns, 3.6 * rows), squeeze=False)
    for axis, spec in zip(axes.ravel(), panels, strict=False):
        panel(axis, modulation, *spec)
    for axis in axes.ravel()[len(panels) :]:
        axis.set_visible(False)
    handles, labels = axes[0][0].get_legend_handles_labels()
    figure.legend(handles, labels, loc="upper center", ncol=len(STEPS), bbox_to_anchor=(0.5, 1.05))
    figure.suptitle(f"modulation: {modulation}", y=1.10, fontsize=12)
    return figure


LEARNING_CURVES = [
    ("val/loss", "Validation loss"),
    ("val/auroc", "Validation AUROC"),
    ("val/bacc", "Validation balanced accuracy"),
    ("val/hidden_min_auroc", "Worst-group AUROC"),
    ("val/hidden_min_bacc", "Worst-group balanced accuracy"),
    ("val/hidden_auroc_gap", "Cohort AUROC spread (max - min)"),
]

## learning curves — fc

In [ ]:
figure = figure_for("fc", LEARNING_CURVES)
figure.savefig(FIGURES / "learning_curves_fc.png")

## learning curves — stage4+fc

In [ ]:
figure = figure_for("stage4+fc", LEARNING_CURVES)
figure.savefig(FIGURES / "learning_curves_stage4_fc.png")

## GroupDRO が動いたか

`weight_entropy` の上限は一様分布の log(10) = 2.3026。ここから離れるほど、特定の cohort へ
重みが寄っている。判定は「stage 内で寝るか（均衡）、下がり続けるか（未収束）」で行う。

In [ ]:
GROUP_DRO = [
    ("train/group_dro/weight_entropy", "Weight entropy", float(np.log(10)), "log(10)"),
    ("train/group_dro/max_q", "Max q", 0.1, "uniform"),
]

for modulation in MODULATIONS:
    figure = figure_for(modulation, GROUP_DRO, columns=2)
    figure.savefig(FIGURES / f"group_dro_{modulation.replace('+', '_')}.png")

## q は何に寄ったのか

cohort は stage ごとに引き直されるので、群を stage 間で追うことはできない。代わりに
**分析単位を `(run, stage, cohort)` とし、stage の中だけで `q` が何と相関するかを見る**。
知りたいのは「群 k がどうなったか」ではなく「DRO が何を難しさと見なしたか」なので、
stage 内で閉じた問いとして立てられる。4 run × 2 stage = 8 回の再抽選が、そのまま反復サンプルになる。

`assignments.parquet` は train / val / test を同じ `group_id` で持つので、train 側の `q_k` と
val 側の `hidden_auroc_k` は同じクラスタを指す。

対立仮説は 3 つ。

1. **重み由来** — `q` が陽性 class weight の大きい群に寄る。`weighting=inverse` では class weight が
   群の陽性率の決定的な関数なので、これは「陽性率の低い群に寄る」と同義
2. **難しさ由来** — `q` が AUROC の低い群に寄る。想定どおり
3. **サイズ由来** — `q` が小さい群に寄る。少数群は loss の分散が大きい

In [ ]:
groups = pd.read_csv(RESULTS / "cohort_groups.csv")

PAIRS = {
    "q ~ positive weight": "positive_weight",
    "q ~ val AUROC": "val_auroc",
    "q ~ train size": "train_size",
}
correlation = pd.DataFrame(
    [
        {
            "condition": condition,
            "stage": stage,
            **{name: data["q"].corr(data[column], method="spearman") for name, column in PAIRS.items()},
            "q spread": data["q"].max() - data["q"].min(),
        }
        for (condition, stage), data in groups.groupby(["condition", "stage"])
    ]
)
correlation.round(3)

## 全 cohort の推移

上の top3 / bottom3 は stage 最終 epoch の `q` で切った区分で、epoch ごとの順位変動を潰している。
圧縮せずに 10 群すべてを描く。

群の色は **stage 最終 epoch の `q` の順位**で決める（濃いほど `q` が大きい）。同じ図の中で同じ色は
同じ群を指すので、`q` の panel で上に行く線が AUROC の panel でどう動くかを追える。順位そのものは
色が表すので凡例は置かない。stage をまたぐと cohort が変わるため、色の対応も stage 内で閉じる。

群ごとの loss は今回の run には記録がない（`hidden_max_loss` と `hidden_loss_gap` のみ）。
`hidden_cohort_logger.py` に `hidden_loss_{k}` の記録を足したので、次の run からは同じ形で描ける。

In [ ]:
import matplotlib.cm as cm

GROUP_PANELS = [
    ("train/group_dro/q_{:02d}", "GroupDRO weight q", 0.1, "uniform"),
    ("val/hidden_auroc_{:02d}", "Validation AUROC per cohort", None, ""),
    ("val/hidden_support_{:02d}", "Validation support", None, ""),
]


def all_groups(modulation, step):
    """10 群すべての推移を stage ごとに描く。色は stage 最終 epoch の q の順位。

    Args:
        modulation: 描く変調範囲
        step: 描く step size

    Returns:
        Figure: 描画した figure
    """
    stages = sorted(groups["stage"].unique())
    figure, axes = plt.subplots(len(stages), len(GROUP_PANELS), figsize=(4.6 * len(GROUP_PANELS), 3.5 * len(stages)), squeeze=False)
    for row, stage in enumerate(stages):
        ranked = groups[(groups["modulation"] == modulation) & (groups["step_size"] == step) & (groups["stage"] == stage)].sort_values("q")
        # q の小さい群を薄く、大きい群を濃くする。順位は 1 つの sequential ramp で表す。
        shade = {group: cm.YlGnBu(0.25 + 0.7 * index / (len(ranked) - 1)) for index, group in enumerate(ranked["group"])}
        epochs = frame[(frame["modulation"] == modulation) & (frame["step_size"] == step) & (frame["stage"] == stage)]
        for column, (template, title, reference, reference_label) in enumerate(GROUP_PANELS):
            axis = axes[row][column]
            if reference is not None:
                axis.axhline(reference, color=MUTED, linewidth=0.8, linestyle="--", zorder=1)
                axis.text(epochs["epoch"].max() + 0.05, reference, reference_label, color=MUTED, fontsize=8, va="center")
            for group in ranked["group"]:
                axis.plot(epochs["epoch"], epochs[template.format(group)], color=shade[group], linewidth=1.4, zorder=3)
            axis.set_title(f"{title} — {stage}")
            axis.set_xlabel("Stage epoch")
    figure.suptitle(f"modulation: {modulation} / step {step:g}   (color: darker = larger q)", y=1.02, fontsize=12)
    return figure


for modulation in MODULATIONS:
    for step in STEPS:
        figure = all_groups(modulation, step)
        figure.savefig(FIGURES / f"all_groups_{modulation.replace('+', '_')}_step{step:g}.png")